In [1]:
from dotenv import load_dotenv

load_dotenv()
print("Notebook ready")

Notebook ready


In [2]:
%pip install anthropic

Note: you may need to restart the kernel to use updated packages.


In [3]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-5"

In [ ]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


# Chat function with temperature parameter
def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        #temperature is deprecated for new models
        #"temperature": temperature,
        
    }
    
    if system:
        params["system"] = system
    
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    message = client.messages.create(**params)
    # Claude Sonnet 5 may return ThinkingBlock(s) before the TextBlock
    text_parts = [
        block.text for block in message.content
        if getattr(block, "type", None) == "text"
    ]
    return "".join(text_parts)



## BASIC EXAMPLE

In [14]:
# Start with an empty message list
messages = []

# Add the initial user question
add_user_message(messages, "Define quantum computing in one sentence")

# Get Claude's response
answer = chat(messages)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

# Add a follow-up question
add_user_message(messages, "Write another sentence")

# Get the follow-up response with full context
final_answer = chat(messages)

print 
final_answer

'Unlike classical bits, which represent either a 0 or a 1, quantum bits (qubits) can exist in a combination of both states simultaneously, allowing quantum computers to explore many possible solutions at once.'

## SYSTEM PROMPT EXAMPLE

In [15]:

messages = []

# With system prompt
system = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""

# Add the initial user question
add_user_message(messages, "Which is the area of a circle with radius 5?")


answer = chat(messages, system=system)

# Get Claude's response
answer = chat(messages)

print(answer)

# Area of a Circle with Radius 5

The formula for the area of a circle is:

$$A = \pi r^2$$

**Calculation:**
$$A = \pi (5)^2 = 25\pi$$

**Result:**
$$A = 25\pi \approx 78.54 \text{ square units}$$


In [7]:
## EXAMPLE WITH TEMPERATURE

In [13]:
messages = []

# With system prompt
system = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""

# Add the initial user question
add_user_message(messages, "Which is the area of a circle with radius 5?")

# Low temperature - more predictable // TEMPERATURE IS DEPRICATED FOR NEW MODELS
answer = chat(messages, system=system, temperature=0.0)

print(answer)

messages = []

# Add the initial user question
add_user_message(messages, "Which is the area of a circle with radius 5?")

# High temperature - more creative  
answer = chat(messages, system=system, temperature=1.0)

print(answer)

I'd be happy to help you work through this problem!

Let's start with what you already know:

- What's the formula for the area of a circle in terms of its radius?

Once you tell me that, we can plug in the radius value together and work out the answer step by step.
I'd be happy to guide you through this!

To find the area of a circle, do you remember the formula that relates area to the radius?

Think about it — it involves a special constant (π) and the radius in some way. What do you recall?


In [ ]:
## STREAMING EXAMPLE

In [ ]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        # Send each chunk to your client
        pass
    
    # Get the complete message for database storage
    final_message = stream.get_final_message()

    print(final_message)

ParsedMessage(id='msg_011CdnXETkHe2r72tR9WhxZ6', container=None, content=[ParsedTextBlock(citations=None, text='A lightweight, cloud-native database called "Nimbustore" that promises infinite horizontal scaling, real-time sync across devices, and zero-downtime schema migrations, all through a simple JSON-based query language.', type='text', parsed_output=None)], model='claude-sonnet-5', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=21, output_tokens=72, output_tokens_details=None, server_tool_use=None, service_tier='standard'))


## STRUCTURED DATA EXAMPLE

In [17]:
messages = []

add_user_message(messages, "Generate a pure JSON object to indicate the status of the weatherm, no aditional characters or text.")
# Assistant message is not supported in the new models
#add_assistant_message(messages, "```json")
text = chat(messages,stop_sequences=["```"])

text

''